# AlphaGalerkin Multi-Physics Demonstrations

This notebook demonstrates AlphaGalerkin's **resolution-independent** neural operator capabilities across multiple physical domains:

1. **Electromagnetism**: Poisson equation (existing)
2. **Heat Transfer**: Heat equation (time evolution)
3. **Fluid Dynamics**: Darcy flow in porous media
4. **Structural Mechanics**: Linear elasticity (vector fields)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# Physics solvers
from src.physics.poisson import PoissonSolver
from src.physics.heat import HeatSolver
from src.physics.darcy import DarcyFlowSolver
from src.physics.elasticity import ElasticitySolver

# Visualization setup
plt.style.use('default')
plt.rcParams['figure.dpi'] = 100

# Configure logging
import structlog
structlog.configure(
    processors=[structlog.dev.ConsoleRenderer()],
    wrapper_class=structlog.BoundLogger,
    context_class=dict,
    logger_factory=structlog.PrintLoggerFactory(),
)

print("Multi-Physics Solvers loaded successfully!")

## 1. Electromagnetism: Poisson Equation

Solves $\nabla^2 \phi = \rho$ for electrostatic potential.

In [ ]:
# Generate Poisson sample at different resolutions
resolutions = [16, 32, 64]

fig, axes = plt.subplots(len(resolutions), 2, figsize=(10, 4*len(resolutions)))
fig.suptitle('Poisson Equation: Charge Density → Potential', fontsize=14)

for i, res in enumerate(resolutions):
    solver = PoissonSolver(resolution=res)
    sample = solver.generate_sample(seed=42)
    
    # Reshape for visualization
    charges = sample.charges.reshape(res, res)
    potential = sample.potential.reshape(res, res)
    
    axes[i, 0].imshow(charges, cmap='RdBu', origin='lower')
    axes[i, 0].set_title(f'Charge Density ({res}x{res})')
    axes[i, 0].set_xlabel('x')
    axes[i, 0].set_ylabel('y')
    
    axes[i, 1].imshow(potential, cmap='viridis', origin='lower')
    axes[i, 1].set_title(f'Potential ({res}x{res})')
    axes[i, 1].set_xlabel('x')
    axes[i, 1].set_ylabel('y')

plt.tight_layout()
plt.show()

## 2. Heat Transfer: Heat Equation

Solves $\frac{\partial u}{\partial t} = \alpha \nabla^2 u$ (diffusion of temperature).

In [ ]:
# Heat equation demonstration
fig, axes = plt.subplots(len(resolutions), 2, figsize=(10, 4*len(resolutions)))
fig.suptitle('Heat Equation: Initial Temperature → Final Temperature', fontsize=14)

for i, res in enumerate(resolutions):
    solver = HeatSolver(resolution=res, alpha=0.01, total_time=1.0)
    sample = solver.generate_sample(seed=42)
    
    u0 = sample.input_field.reshape(res, res)
    u_final = sample.output_field.reshape(res, res)
    
    # Same colorbar range for comparison
    vmin, vmax = min(u0.min(), u_final.min()), max(u0.max(), u_final.max())
    norm = Normalize(vmin=vmin, vmax=vmax)
    
    axes[i, 0].imshow(u0, cmap='hot', origin='lower', norm=norm)
    axes[i, 0].set_title(f'Initial (t=0, {res}x{res})')
    axes[i, 0].set_xlabel('x')
    axes[i, 0].set_ylabel('y')
    
    im = axes[i, 1].imshow(u_final, cmap='hot', origin='lower', norm=norm)
    axes[i, 1].set_title(f'Final (t=1, {res}x{res})')
    axes[i, 1].set_xlabel('x')
    axes[i, 1].set_ylabel('y')

plt.tight_layout()
plt.show()

print("Notice how the temperature smooths out (diffuses) over time.")

## 3. Fluid Dynamics: Darcy Flow

Solves $-\nabla \cdot (a(x) \nabla u(x)) = f(x)$ for pressure in porous media.

In [ ]:
# Darcy flow demonstration
fig, axes = plt.subplots(len(resolutions), 2, figsize=(10, 4*len(resolutions)))
fig.suptitle('Darcy Flow: Permeability Field → Pressure Field', fontsize=14)

for i, res in enumerate(resolutions):
    solver = DarcyFlowSolver(resolution=res, forcing=1.0)
    sample = solver.generate_sample(seed=42)
    
    permeability = sample.input_field.reshape(res, res)
    pressure = sample.output_field.reshape(res, res)
    
    axes[i, 0].imshow(np.log10(permeability), cmap='coolwarm', origin='lower')
    axes[i, 0].set_title(f'Log Permeability ({res}x{res})')
    axes[i, 0].set_xlabel('x')
    axes[i, 0].set_ylabel('y')
    
    axes[i, 1].imshow(pressure, cmap='Blues', origin='lower')
    axes[i, 1].set_title(f'Pressure ({res}x{res})')
    axes[i, 1].set_xlabel('x')
    axes[i, 1].set_ylabel('y')

plt.tight_layout()
plt.show()

print("High permeability regions allow easier flow (lower pressure gradient).")

## 4. Structural Mechanics: Linear Elasticity

Solves $\nabla \cdot \sigma + F = 0$ for displacement under body forces.

In [ ]:
# Elasticity demonstration (vector field)
fig, axes = plt.subplots(len(resolutions), 3, figsize=(15, 4*len(resolutions)))
fig.suptitle('Linear Elasticity: Body Force → Displacement', fontsize=14)

for i, res in enumerate(resolutions):
    solver = ElasticitySolver(resolution=res, young_modulus=1.0, poisson_ratio=0.3)
    sample = solver.generate_sample(seed=42)
    
    # Force and displacement are vector fields
    F = sample.input_field.reshape(res, res, 2)
    u = sample.output_field.reshape(res, res, 2)
    
    # Force magnitude
    F_mag = np.sqrt(F[..., 0]**2 + F[..., 1]**2)
    axes[i, 0].imshow(F_mag, cmap='Reds', origin='lower')
    axes[i, 0].set_title(f'Force Magnitude ({res}x{res})')
    axes[i, 0].set_xlabel('x')
    axes[i, 0].set_ylabel('y')
    
    # Displacement magnitude
    u_mag = np.sqrt(u[..., 0]**2 + u[..., 1]**2)
    axes[i, 1].imshow(u_mag, cmap='Greens', origin='lower')
    axes[i, 1].set_title(f'Displacement Magnitude ({res}x{res})')
    axes[i, 1].set_xlabel('x')
    axes[i, 1].set_ylabel('y')
    
    # Quiver plot of displacement vectors (subsampled)
    step = max(1, res // 8)
    X, Y = np.meshgrid(np.arange(0, res, step), np.arange(0, res, step))
    axes[i, 2].quiver(X, Y, u[::step, ::step, 0], u[::step, ::step, 1], scale=1.0)
    axes[i, 2].set_title(f'Displacement Vectors ({res}x{res})')
    axes[i, 2].set_xlim(-1, res)
    axes[i, 2].set_ylim(-1, res)
    axes[i, 2].set_aspect('equal')

plt.tight_layout()
plt.show()

print("Vector field output demonstrates handling of multi-output problems.")

## Resolution Independence Summary

All solvers generate consistent physics at different resolutions — a key property for neural operator training.

In [ ]:
print("=" * 60)
print("AlphaGalerkin Multi-Physics Solvers Summary")
print("=" * 60)

solvers = [
    ("Poisson", PoissonSolver),
    ("Heat", HeatSolver),
    ("Darcy", DarcyFlowSolver),
    ("Elasticity", ElasticitySolver),
]

for name, SolverClass in solvers:
    solver = SolverClass(resolution=32)
    sample = solver.generate_sample(seed=0)
    print(f"\n{name}:")
    print(f"  Input shape:  {sample.input_field.shape}")
    print(f"  Output shape: {sample.output_field.shape}")
    print(f"  Grid size:    {sample.grid_size}")
    if sample.metadata:
        print(f"  Metadata:     {sample.metadata}")